# Few-Shot Comparison — LLMs vs ArchCodeT5-Evo

**Goal:** compare several recent LLMs in a 1-shot setting on the ADL evolution
task, to justify the benefit of the specialized fine-tuning approach used in
`Evolution_Finetuning.ipynb`.

**Models compared (via the Groq API free tier):**

| # | Model | Params | Type |
|---|-------|--------|------|
| 1 | Llama-3.1-8B-Instant | 8B | general-purpose LLM |
| 2 | Llama-3.3-70B-Versatile | 70B | general-purpose LLM |
| 3 | Llama-4-Scout-17B-16E | 17B×16E | general-purpose LLM |
| 4 | GPT-OSS-20B | 20B | reasoning LLM |
| 5 | GPT-OSS-120B | 120B | reasoning LLM |
| 6 | Qwen3-32B | 32B | reasoning LLM |



**Protocol:** 1-shot prompting, evaluated on a stratified random subset of
20 test examples (seed=42), compared against the fine-tuned pipeline
(`CodeT5-base`, Strategy A) evaluated on the full test set.



## Step 0 — Installation & API Keys

In [ ]:
# ============================================================
# STEP 0A — Install dependencies
# ============================================================
!pip install groq rouge_score bert_score nltk -q

import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

print('✅ Installation complete')

In [ ]:
# ============================================================
# STEP 0B — API key
# ============================================================
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    import getpass
    GROQ_API_KEY = getpass.getpass('Enter your Groq API key: ')

for name, key in [('Groq', GROQ_API_KEY)]:
    if not key:
        print(f'❌ {name}: missing key!')
    else:
        print(f'✅ {name}: {key[:8]}...{key[-4:]} ({len(key)} chars)')

In [ ]:
# ============================================================
# STEP 0C — Initialize the Groq client
#
# NOTE: in the original notebook this initialization lived in a cell
# placed AFTER the inference cells that already called the Groq API
# — running the notebook top to bottom would crash with a NameError
# on `groq_client`. It is moved here, right after the API key is
# loaded, so the notebook runs linearly.
# ============================================================
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ groq_client initialized")

In [ ]:
# ============================================================
# STEP 0D — Google Drive & paths
# ============================================================
from google.colab import drive
import json, os, time, re
import numpy as np

drive.mount('/content/drive')

# NOTE: update BASE_DIR if your dataset lives elsewhere.
BASE_DIR  = '/content/drive/MyDrive'
EVO_DIR   = f'{BASE_DIR}/dataset_clean/evolution'
EVO_TRAIN = f'{EVO_DIR}/evo_train.jsonl'
EVO_TEST  = f'{EVO_DIR}/evo_test_v2.jsonl'

# Path to the fine-tuned pipeline's own results (Model 1 from
# Evolution_Finetuning.ipynb), used later for the comparison table.
PIPELINE_RESULTS_PATH = f'{BASE_DIR}/evo_CodeT5base_fullFT/results_CodeT5base_StratA_FullFT.json'

for label, path in [('evo_train', EVO_TRAIN), ('evo_test', EVO_TEST),
                     ('pipeline results', PIPELINE_RESULTS_PATH)]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {label:<20} {path}')

##  Step 1 — Dataset Loading

In [ ]:
# ============================================================
# STEP 1A — Load the JSONL files
# ============================================================
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(EVO_TRAIN)
test_data  = load_jsonl(EVO_TEST)

print(f'📊 Train: {len(train_data)} examples')
print(f'📊 Test : {len(test_data)} examples')

In [ ]:
# ============================================================
# STEP 1B — Stratified selection of the test subset (n=20)
# Stratified by operation type to preserve the distribution of
# the full test set — seed=42 for reproducibility.
#
# Justification for the paper: "Baseline LLMs are evaluated on a
# stratified subset of 20 test samples selected with seed=42 for
# reproducibility. The subset preserves the operation-type
# distribution of the full test set (ADD/REMOVE/MODIFY/REPLACE/
# CONNECT/DISCONNECT)."
# ============================================================
import random
from collections import defaultdict, Counter

random.seed(42)

# Group examples by operation
test_examples_by_op = defaultdict(list)
for ex in test_data:
    op = ex.get('operation',
         ex['input'].split('[OP]')[1].split('[')[0].strip())
    test_examples_by_op[op].append(ex)

n_ops       = len(test_examples_by_op)   # 6 operations
n_total     = 20
n_per_op    = n_total // n_ops           # 3 per operation
n_remainder = n_total % n_ops            # 2 leftover -> most frequent ops

# Sort by descending frequency to distribute the remainder
ops_sorted = sorted(test_examples_by_op.items(), key=lambda x: -len(x[1]))

test_subset = []
for i, (op, examples) in enumerate(ops_sorted):
    quota = n_per_op + (1 if i < n_remainder else 0)
    quota = min(quota, len(examples))   # safety net if an op is under-represented
    test_subset.extend(random.sample(examples, quota))

random.shuffle(test_subset)   # shuffle to avoid any ordering bias

print(f'✅ test_subset: {len(test_subset)} examples')
print(f'   Distribution:')
ops_subset = Counter(
    ex.get('operation', ex['input'].split('[OP]')[1].split('[')[0].strip())
    for ex in test_subset
)
for op, n in sorted(ops_subset.items()):
    print(f'   {op:<30} : {n}')

# ── Sanity check: confirm the subset preserves the full distribution ──
ops_full = Counter(ex.get('operation', ex['input'].split('[OP]')[1].split('[')[0].strip())
                    for ex in test_data)
print(f'\nFull test distribution    : {dict(ops_full)}')
print(f'Subset distribution       : {dict(ops_subset)}')

In [ ]:
# ============================================================
# STEP 1C — Select the few-shot example (from the training set)
# Strategy: an ADD_COMPONENT example (the most frequent operation)
# selected reproducibly (seed=42)
# ============================================================
from collections import defaultdict

train_examples_by_op = defaultdict(list)
for ex in train_data:
    op = ex.get('operation', 'UNKNOWN')
    train_examples_by_op[op].append(ex)

print('📋 Operation distribution (train set):')
for op, items in sorted(train_examples_by_op.items(), key=lambda x: -len(x[1])):
    print(f'  {op:<30} : {len(items)} examples')

# Most frequent operation -> most representative few-shot example
dominant_op = max(train_examples_by_op, key=lambda op: len(train_examples_by_op[op]))

random.seed(42)   # reproducible -> same example on every run
FEW_SHOT_EXAMPLES = [random.choice(train_examples_by_op[dominant_op])]

print(f'\n✅ Few-shot example selected')
print(f'   Operation : {dominant_op}')
print(f'   Input     : {str(FEW_SHOT_EXAMPLES[0]["input"])[:80]}...')
print(f'   Target    : {str(FEW_SHOT_EXAMPLES[0]["target"])[:80]}...')

##Step 2 — Prompt Construction

In [ ]:
# ============================================================
# STEP 2 — 1-shot prompt with truncation
# ============================================================
SYSTEM_PROMPT = (
    "You are an expert in Architecture Description Languages (ADL), "
    "specifically ACME and AADL. Your task is to apply architectural evolution "
    "operations to ADL specifications. Given an ADL architecture and a natural "
    "language evolution request, generate the evolved ADL architecture. "
    "Output ONLY the evolved ADL code, nothing else."
)

MAX_CHARS = 1500  # 800 was too short -> truncated the ADL mid-block
                   # 1500 ≈ 375 tokens: reasonable for the Groq free tier

def truncate(text, max_c=MAX_CHARS):
    if len(text) <= max_c:
        return text
    # Clean truncation: cut at the last full line before max_c
    truncated  = text[:max_c]
    last_break = truncated.rfind('\n')
    if last_break > max_c * 0.7:   # only cut on a line break if we keep 70%+ of the text
        return text[:last_break] + '\n...'
    return truncated + '...'

def build_prompt(few_shot_examples, test_input):
    """1-shot prompt with clean line-based truncation."""
    ex = few_shot_examples[0]
    prompt  = "### Example\n"
    prompt += f"Input:\n{truncate(str(ex['input']))}\n"
    prompt += f"Output:\n{truncate(str(ex['target']))}\n\n"
    prompt += "### Now apply the evolution:\n"
    prompt += f"Input:\n{truncate(str(test_input))}\n"
    prompt += "Output:"
    return prompt

# ── Sanity check ──────────────────────────────────────────
test_prompt          = build_prompt(FEW_SHOT_EXAMPLES, test_subset[0]['input'])
estimated_tokens      = len(test_prompt) // 4
tokens_for_20_examples = 20 * estimated_tokens
groq_token_limit       = 6_000   # free tier: ~6k tokens/min

print(f'📏 Prompt length        : {len(test_prompt)} chars')
print(f'📏 Estimated tokens      : ~{estimated_tokens} / call')
print(f'📏 Total for 20 examples : ~{tokens_for_20_examples:,} tokens')
print(f'📏 Groq free tier limit  : ~{groq_token_limit:,} tokens/min')
if tokens_for_20_examples > groq_token_limit * 5:
    print('⚠️  Rate-limit risk — consider adding time.sleep(1) between calls')
else:
    print('✅ Volume compatible with the Groq free tier')

# Preview of the final prompt
print(f'\n{"─"*55}')
print('📄 Prompt preview (first 500 chars):')
print(test_prompt[:500])
print('...')

## Step 3 — Baseline Inference (Groq API)

> **Note on this section:** the original notebook accumulated several
> rounds of live, interactive patching (cells named `FIX`, `INIT`,
> `EXTRA`, `FULL-RERUN`,  `RESTORE`...) as models were added
> one at a time across several Colab sessions. Some of those cells
> produced an earlier, incomplete run whose numbers differ slightly
> from the final reported results (see the reproducibility caveat in
> the introduction). The cells below consolidate all of that into a
> single, linear pipeline that runs all 6 models in one pass.

In [ ]:
# ============================================================
# Baseline model configuration
# ============================================================

BASELINE_MODELS = {
    "Llama-3.1-8B"      : "llama-3.1-8b-instant",
    "Llama-3.3-70B"     : "llama-3.3-70b-versatile",
    "Llama-4-Scout-17B" : "meta-llama/llama-4-scout-17b-16e-instruct",
    "GPT-OSS-20B"       : "openai/gpt-oss-20b",
    "GPT-OSS-120B"      : "openai/gpt-oss-120b",
    "Qwen3-32B"         : "qwen/qwen3-32b",
}

# Reasoning models need a larger max_tokens budget (they "think"
# before answering, which consumes part of the token budget).
REASONING_MODELS = {
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "qwen/qwen3-32b",
}

print("✅ Baseline configuration loaded")
for name, mid in BASELINE_MODELS.items():
    tag = "🧠 reasoning" if mid in REASONING_MODELS else "💬 standard"
    print(f"   {tag}  {name:<20} -> {mid}")

In [ ]:
# ============================================================
# Evaluation metrics for one model's predictions
# ============================================================
from rouge_score import rouge_scorer as rs_module
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score_fn

def compute_metrics(predictions: list, references: list, model_name: str) -> dict:
    """Aggregate NLP metrics (EM, BLEU, ROUGE-1/2/L, METEOR, BERT-F1)."""
    preds = [p.strip() if p else "" for p in predictions]
    refs  = [r.strip() if r else "" for r in references]

    em = sum(1 if p == r else 0 for p, r in zip(preds, refs)) / len(preds)

    smooth = SmoothingFunction().method1
    refs_bleu  = [[r.split()] for r in refs]
    preds_bleu = [p.split()   for p in preds]
    bleu = corpus_bleu(refs_bleu, preds_bleu, smoothing_function=smooth)

    scorer_r = rs_module.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = scorer_r.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    rouge1, rouge2, rougeL = float(np.mean(r1)), float(np.mean(r2)), float(np.mean(rL))

    meteor_scores = []
    for p, r in zip(preds, refs):
        try:    meteor_scores.append(meteor_score([r.split()], p.split()))
        except: meteor_scores.append(0.0)
    meteor = float(np.mean(meteor_scores))

    _, _, F1 = bert_score_fn(preds, refs, lang="en",
                              rescale_with_baseline=False, verbose=False)
    bert_f1 = float(F1.mean())

    metrics = {
        "model"  : model_name,
        "EM"     : em,
        "BLEU"   : bleu,
        "ROUGE-1": rouge1,
        "ROUGE-2": rouge2,
        "ROUGE-L": rougeL,
        "METEOR" : meteor,
        "BERT-F1": bert_f1,
    }

    print(f"📊 {model_name}")
    for k in ["EM", "BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR", "BERT-F1"]:
        print(f"   {k:<11} : {metrics[k]:.4f}")

    return metrics

print("✅ compute_metrics defined")

In [ ]:
# ============================================================
# Unified call function (handles reasoning models, rate limits,
# decommissioned models)
# ============================================================
import re, time

def call_groq_model(test_input, few_shot_examples, model_id):
    """
    Calls any Groq-hosted model. Automatically handles:
      - reasoning models (empty `content` -> fall back to `reasoning`)
      - 429 rate limiting, with automatic wait
      - decommissioned models
    """
    prompt  = build_prompt(few_shot_examples, test_input)
    max_tok = 1024 if model_id in REASONING_MODELS else 512

    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tok,
                temperature=0.0,
            )
            msg     = response.choices[0].message
            content = (msg.content or "").strip()

            # Reasoning fallback: content empty after internal reasoning
            if not content and hasattr(msg, "reasoning") and msg.reasoning:
                lines   = [l.strip() for l in msg.reasoning.strip().splitlines() if l.strip()]
                content = lines[-1] if lines else ""
                if content:
                    print(f"  ⚠️  [{model_id}] empty content -> used reasoning fallback")

            return content

        except Exception as e:
            err = str(e)
            if "429" in err:
                match = re.search(r"try again in (\d+)m", err)
                wait  = int(match.group(1)) * 60 + 30 if match else 60
                print(f"  ⏳ Rate limit — waiting {wait//60}m{wait%60}s...")
                time.sleep(wait)
            elif any(k in err for k in ["decommissioned", "deprecated", "model_not_found"]):
                print(f"  ❌ Model unavailable: {model_id}")
                return ""
            else:
                print(f"  ⚠️  Unexpected error ({model_id}), attempt {attempt+1}/3: {e}")
                if attempt == 2:
                    return ""
    return ""

print("✅ call_groq_model defined")

In [ ]:
# ============================================================
# Run 1-shot inference and compute metrics for all 6 baselines
#
# Skips any model already present in `predictions_baselines`, so this
# cell can be safely re-run if the runtime disconnects partway through.
# ============================================================
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

if "predictions_baselines" not in globals():
    predictions_baselines = {}
if "metriques_baselines" not in globals():
    metriques_baselines = {}

refs = [ex["target"] for ex in test_subset]

for name, mid in BASELINE_MODELS.items():

    if name in predictions_baselines:
        print(f"⏭️  {name:<26} already computed — skipping")
        continue

    print(f"\n{'='*58}")
    print(f"🔄  {name}  ({mid})")
    print(f"{'='*58}")

    preds  = []
    errors = 0

    for i, ex in enumerate(test_subset):
        print(f"  [{i+1:02d}/{len(test_subset)}]", end="\r")
        pred = call_groq_model(ex["input"], FEW_SHOT_EXAMPLES, mid)
        preds.append(pred)
        if not pred:
            errors += 1
        time.sleep(0.8)   # stay within the free-tier rate limit

    predictions_baselines[name] = preds
    success_rate = (len(preds) - errors) / len(preds) * 100
    print(f"\n  ✅ {len(preds)} predictions  |  "
          f"successful: {len(preds)-errors}/{len(preds)} ({success_rate:.0f}%)")

    # Compute metrics for this model right away, so partial progress
    # survives a runtime disconnect.
    metriques_baselines[name] = compute_metrics(preds, refs, name)

# ── Recap ───────────────────────────────────────────────────
print(f"\n{'='*58}")
print(f"✅ Final state:")
print(f"   predictions_baselines : {len(predictions_baselines)} models")
print(f"   metriques_baselines   : {len(metriques_baselines)} models")
for name, m in sorted(metriques_baselines.items(), key=lambda x: -x[1]['BERT-F1']):
    print(f"   {name:<26} EM={m['EM']:.4f}  BERT-F1={m['BERT-F1']:.4f}")

## Step 4 — Statistical Comparison vs the Fine-Tuned Pipeline

In [ ]:
# ============================================================
# Bootstrap 95% CI on all baselines
# Required for the paper given the small sample size (n=20).
# B=2000 iterations — stable and fast on CPU.
# ============================================================
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score_fn

def bootstrap_ci95(predictions, references, B=2000, seed=42):
    """
    Percentile bootstrap 95% CI on EM, BLEU, ROUGE-L, METEOR, BERT-F1.
    Returns a dict mapping metric -> (mean, lower, upper).
    """
    rng   = np.random.default_rng(seed)
    n     = len(predictions)
    preds = [p.strip() if p else "" for p in predictions]
    refs  = [r.strip() if r else "" for r in references]

    # Precompute per-example BERT-F1 (avoids B*n API/model calls)
    _, _, F1 = bert_score_fn(preds, refs, lang="en",
                              rescale_with_baseline=False, verbose=False)
    bert_f1_per = F1.numpy()

    # Precompute per-example ROUGE-L
    scorer_r = rs_module.RougeScorer(['rougeL'], use_stemmer=True)
    rougeL_per = np.array([
        scorer_r.score(r, p)['rougeL'].fmeasure
        for p, r in zip(preds, refs)
    ])

    # Precompute per-example METEOR
    meteor_per = []
    for p, r in zip(preds, refs):
        try:
            meteor_per.append(meteor_score([r.split()], p.split()))
        except Exception:
            meteor_per.append(0.0)
    meteor_per = np.array(meteor_per)

    # Precompute per-example EM
    em_per = np.array([float(p == r) for p, r in zip(preds, refs)])

    # ── Bootstrap ──────────────────────────────────────────
    boot = {"EM": [], "BLEU": [], "ROUGE-L": [], "METEOR": [], "BERT-F1": []}
    smooth = SmoothingFunction().method1

    for _ in range(B):
        idx = rng.integers(0, n, size=n)

        boot["EM"].append(em_per[idx].mean())

        # BLEU: recomputed on the resampled subset (needs the full corpus)
        refs_b  = [[refs[i].split()] for i in idx]
        preds_b = [preds[i].split()  for i in idx]
        boot["BLEU"].append(
            corpus_bleu(refs_b, preds_b, smoothing_function=smooth)
        )

        boot["ROUGE-L"].append(rougeL_per[idx].mean())
        boot["METEOR"].append(meteor_per[idx].mean())
        boot["BERT-F1"].append(bert_f1_per[idx].mean())

    results = {}
    for metric, values in boot.items():
        arr = np.array(values)
        results[metric] = (
            float(arr.mean()),
            float(np.percentile(arr, 2.5)),
            float(np.percentile(arr, 97.5))
        )
    return results

# ── Compute for all baselines ──────────────────────────────
print("⏳ Running bootstrap 95% CI (B=2000)...\n")

ic95_baselines = {}
for name, preds in predictions_baselines.items():
    print(f"  🔄 {name:<26}", end=" ")
    ic = bootstrap_ci95(preds, refs, B=2000)
    ic95_baselines[name] = ic
    print(f"✅  EM={ic['EM'][0]:.4f} [{ic['EM'][1]:.4f}-{ic['EM'][2]:.4f}]")

print("\n✅ Bootstrap complete")

In [ ]:
# ============================================================
# Load the fine-tuned pipeline's results (Model 1, CodeT5-base
# Strategy A, from Evolution_Finetuning.ipynb) for comparison.
#
# Loaded directly from that notebook's saved JSON when available, so
# the two notebooks stay in sync; falls back to the literal values
# reported in the paper if the file cannot be found (e.g. this
# notebook is run standalone, without having run the other one
# first).
# ============================================================
PIPELINE_FALLBACK = {
    "EM"      : 0.3333,
    "BLEU"    : 0.5976,
    "ROUGE-L" : 0.9382,
    "METEOR"  : 0.8974,
    "BERT-F1" : 0.9994,
}

if os.path.exists(PIPELINE_RESULTS_PATH):
    with open(PIPELINE_RESULTS_PATH) as f:
        _pipeline_raw = json.load(f)
    PIPELINE = {k: _pipeline_raw[k] for k in
                ["EM", "BLEU", "ROUGE-L", "METEOR", "BERT-F1"]}
    print(f"✅ Pipeline results loaded from: {PIPELINE_RESULTS_PATH}")
else:
    PIPELINE = PIPELINE_FALLBACK
    print(f"⚠️  {PIPELINE_RESULTS_PATH} not found — using the fallback values")
    print("   (re-run Evolution_Finetuning.ipynb's Model 1 evaluation to refresh them)")

# Pipeline test-set size (full test set, deterministic local inference)
PIPELINE_N_TEST = 120

print(f"   PIPELINE = {PIPELINE}")

In [ ]:
# ============================================================
# Final comparison table — 6 baselines vs the fine-tuned pipeline
# ============================================================
METRICS = ["EM", "BLEU", "ROUGE-L", "METEOR", "BERT-F1"]
SEP     = "  " + "─" * 95
HEAD    = (f"  {'Model':<26} {'Params':>7}  "
           + "  ".join(f"{m:>20}" for m in METRICS))

PARAMS = {
    "Llama-3.1-8B"      : "8B",
    "GPT-OSS-20B"       : "20B",
    "Qwen3-32B"         : "32B",
    "Llama-3.3-70B"     : "70B",
    "Llama-4-Scout-17B" : "17B×16E",
    "GPT-OSS-120B"      : "120B",
}

# Ascending order by parameter count
ORDER = [
    "Llama-3.1-8B",
    "GPT-OSS-20B",
    "Qwen3-32B",
    "Llama-3.3-70B",
    "Llama-4-Scout-17B",
    "GPT-OSS-120B",
]

print()
print("  " + "=" * 95)
print(f"  📊  FEW-SHOT BASELINES (1-shot, n=20) vs ArchCodeT5-Evo (fine-tuned, n={PIPELINE_N_TEST})")
print("  " + "=" * 95)
print(HEAD)
print(SEP)

for name in ORDER:
    if name not in ic95_baselines:
        print(f"  ⚠️  {name} missing from ic95_baselines — re-run the bootstrap cell")
        continue
    ic  = ic95_baselines[name]
    par = PARAMS.get(name, "?")
    cells = []
    for m in METRICS:
        mean, lo, hi = ic[m]
        cells.append(f"{mean:.4f} [{lo:.3f}-{hi:.3f}]")
    print(f"  {name:<26} {par:>7}  "
          + "  ".join(f"{c:>20}" for c in cells))

print(SEP)

# Pipeline row
cells_pipe = [f"* {PIPELINE[m]:.4f}{'':>12}" for m in METRICS]
print(f"  {'* CodeT5-base StratA':<26} {'770M':>7}  "
      + "  ".join(f"{c:>20}" for c in cells_pipe))
print("  " + "=" * 95)
print(f"  * Fine-tuned pipeline, evaluated on the full test set (n={PIPELINE_N_TEST}), deterministic.")
print("  Baselines: 1-shot, n=20, 95% bootstrap percentile CI (B=2000, seed=42).")

# ── Delta vs. the best baseline, per metric ───────────────────
print()
for mt in METRICS:
    best_val  = max(metriques_baselines[n][mt] for n in ORDER if n in metriques_baselines)
    best_name = max((n for n in ORDER if n in metriques_baselines),
                     key=lambda n: metriques_baselines[n][mt])
    delta = PIPELINE[mt] - best_val
    sign  = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
    flag  = "✅" if delta >= 0 else "⚠️ "
    print(f"  {flag} {mt:<8} pipeline={PIPELINE[mt]:.4f}  "
          f"best_baseline={best_val:.4f} ({best_name})  Δ={sign}")

In [ ]:
# ============================================================
# Save the final JSON report (for the paper's results section)
# ============================================================
final_report = {
    "protocol"  : "1-shot, n=20, bootstrap B=2000, seed=42",
    # NOTE: derived from the actual evaluated models rather than the
    # static BASELINE_MODELS config, so this list cannot silently
    # drift out of sync with what was really evaluated.
    "models"    : list(ic95_baselines.keys()),
    "ic95"      : {
        name: {m: list(v) for m, v in ic.items()}
        for name, ic in ic95_baselines.items()
    },
    "pipeline"      : PIPELINE,
    "pipeline_n_test": PIPELINE_N_TEST,
    "note"          : (
        "The pipeline is evaluated on the full test set "
        f"(n={PIPELINE_N_TEST}). Baselines are evaluated on a stratified "
        "subset (n=20) due to API access constraints (Groq free tier)."
    )
}

out_path = f"{EVO_DIR}/fewshot_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False)

print(f"✅ Results saved -> {out_path}")